In [ ]:
import openpyxl
import openpyxl.utils
import builder
import utils as u
import sys
import os
import pandas as pd
from io import BytesIO

In [ ]:
def ProcessInput(input_file):
    # 1. Check Extension
    _, ext = os.path.splitext(input_file)
    ext = ext.lower()

    if ext == '.xlsx':
        #2. Convert to CSV
        try:
            df = pd.read_excel(input_file)
            csv_file = input_file.replace('.xlsx', '_converted.csv')
            df.to_csv(csv_file, index=False)  # write to disk
        except:
            raise ValueError('Invalid Input')
    
    elif ext == '.csv':
        csv_file = input_file

    else:
        raise ValueError('Invalid Input')

    return csv_file


In [ ]:
input_file = '/home/emmatey/code/day-sheet-maker/python/source data/output.csv'
input_file_2 = '/home/emmatey/code/day-sheet-maker/python/source data/_Report Output_Weekly Schedule Report_8000149916_2025-03-22T12_48_14.011.xlsx'
input_file_3 = '/home/emmatey/Documents/Budget.ods'
input_file_4 = '/home/emmatey/Documents/delete-this.xlsx'
csv = ProcessInput(input_file_2)

In [ ]:
hrd = builder.build_store(csv)
column_day_map = u.column_day_map(csv)
_, date_list = column_day_map
weekEndingDate = date_list[-1]
WEEK_ENDING_DATE = weekEndingDate.replace('/', '-')

print(column_day_map)
for dept in hrd.department_list:
    print(dept)


In [ ]:
def FindValidDepts(hrd):
    valid_depts = []
    for dept in hrd.department_list:
        if len(dept.employees) > 0:
            valid_depts.append(dept)
            print(f"Added department: '{dept.dept_name}'")
    return valid_depts

valid_depts = FindValidDepts(hrd)

for dept in valid_depts:
    print(dept)

In [ ]:
def populate_workbook(wb, dept, column_day_map, is_wall: bool = False):
    for day, sheetname in enumerate(wb.sheetnames):
        ws = wb[sheetname]

        # --- Fill Content ---
        employee_group = u.employee_group(dept, day)
        time_blocks = u.TIME_BLOCKS.get(dept.dept_name, [])

        u.insert_title_cell(ws, day, column_day_map)
        u.insert_headers_and_employees(ws, employee_group, day)

        # Special case for Hannaford to Go department
        if dept.dept_name == 'Hannaford to Go':
            u.insert_effective_shopper_table(ws, employee_group, u.EXPEDITOR_REQUIREMENTS, time_blocks, day)
        else:
            u.insert_labor_trackers(ws, employee_group, time_blocks, day)

        # --- Page Setup & Formatting ---
        # Hide columns if "wall" mode
        if is_wall:
            ws.column_dimensions['D'].hidden = True
            ws.column_dimensions['E'].hidden = True
            ws.column_dimensions['F'].hidden = True
            ws.column_dimensions['H'].hidden = True
            ws.page_margins.top = 1.25
            ws.page_setup.orientation = ws.ORIENTATION_PORTRAIT
        else:
            ws.column_dimensions['D'].hidden = False
            ws.column_dimensions['E'].hidden = False
            ws.column_dimensions['F'].hidden = False
            ws.column_dimensions['H'].hidden = False
            ws.page_margins.top = 0.5
            ws.page_setup.orientation = ws.ORIENTATION_LANDSCAPE

        # Margins
        ws.page_margins.left = 0.1
        ws.page_margins.right = 0.1
        ws.page_margins.bottom = 0.25
        ws.page_margins.header = 0.1
        ws.page_margins.footer = 0.1

        # Fit-to-Page
        ws.page_setup.use_fit_to_page = True
        ws.page_setup.fitToWidth = 1
        ws.page_setup.fitToHeight = 1
        ws.page_setup.scale = 100

        # Centering
        ws.print_options.horizontalCentered = True
        ws.print_options.verticalCentered = False

        # Set base column widths (A–M)
        column_widths = {
            'A': 30.0, 'B': 12.0, 'C': 12.0, 'D': 5.0, 'E': 5.0,
            'F': 5.0, 'G': 8.0, 'H': 5.0, 'I': 3.0, 'J': 3.0,
            #'K': 10.0, #'L': 10.0, #'M': 10.0
        }
        for col, width in column_widths.items():
            ws.column_dimensions[col].width = width

        # Customize columns J–M for Hannaford to Go
        if dept.dept_name.lower() == 'hannaford to go':
            ws.column_dimensions['K'].width = 20
            ws.column_dimensions['L'].width = 10
            ws.column_dimensions['M'].width = 5
            ws.column_dimensions['N'].width = 5


        # --- Print Area ---
        last_row = ws.max_row
        last_col_letter = openpyxl.utils.get_column_letter(14)  # Column N = 14
        ws.print_area = f"A1:{last_col_letter}{last_row}"

    # Return the is_wall flag so we know whether to add "_wall" to the filename
    return is_wall

In [ ]:
print(column_day_map)

In [ ]:
def ProcessOutput(save_location_path, valid_depts = valid_depts, column_day_map = column_day_map, WEEK_ENDING_DATE = WEEK_ENDING_DATE):
    # 1. Construct the output folder
    outPath = f'{save_location_path}/daysheets-weekEnding-({WEEK_ENDING_DATE})'
    os.makedirs(outPath, exist_ok = True)

    # 2. Generate both types (table + wall)
    for dept in valid_depts:
        for wall_mode in [False, True]:
            # Load the Excel template fresh each time
            wb = openpyxl.load_workbook('Day Sheet Master.xlsx')
            is_wall = populate_workbook(wb, dept, column_day_map, is_wall=wall_mode)

            # Create file name
            save_name = dept.dept_name.replace(" ", "_")
            if is_wall:
                save_name += "_WALL"

            file_path = os.path.join(outPath, f"{save_name}.xlsx")
            print(f"Saving: {file_path}")
            wb.save(file_path)

    return outPath


In [ ]:
outpath = ProcessOutput('/home/emmatey/Documents/')